# 🪖 Helmet Detection - YOLOv8 Training & NCNN Export Notebook
สมุดบันทึกสำหรับเทรนโมเดลตรวจจับการสวมหมวกกันน็อก (YOLOv8n) บน **Google Colab (GPU T4)**

### 📌 ลำดับขั้นตอน:
1. ตรวจสอบการ์ดจอ (GPU)
2. ติดตั้ง Ultralytics & ไลบรารีที่จำเป็น
3. เตรียมข้อมูล Dataset (`dataset.zip` หรือเชื่อมต่อ Google Drive)
4. เริ่มเทรนโมเดล YOLOv8n (200 Epochs)
5. แสดงผลกราฟความแม่นยำ (mAP, Confusion Matrix)
6. แปลงโมเดลเป็น NCNN Format (`best_ncnn_model.zip`) และดาวน์โหลดลงเครื่อง

## ⚡ ขั้นตอนที่ 1: ตรวจสอบการ์ดจอ (GPU)

In [ ]:
# ตรวจสอบว่าเปิดใช้งาน GPU หรือยัง (ควรเป็น Tesla T4 หรือดีกว่า)
!nvidia-smi

import torch
print("\nCUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ คำเตือน: ยังไม่ได้เปิด GPU ให้ไปที่ Runtime -> Change runtime type -> เลือก T4 GPU")

## 📦 ขั้นตอนที่ 2: ติดตั้ง Ultralytics (YOLOv8)

In [ ]:
!pip install -q ultralytics

import ultralytics
print(f"[OK] Ultralytics version: v{ultralytics.__version__}")

## 📁 ขั้นตอนที่ 3: เตรียม Dataset
*(เลือกวิธีใดวิธีหนึ่ง: โหลดผ่าน Google Drive หรือ อัปโหลดไฟล์ `dataset.zip` ขึ้น Colab)*

In [ ]:
import os
import shutil
import zipfile

extract_dir = "/content/dataset"
dataset_zip = "/content/dataset.zip"

# หากมีไฟล์ dataset.part_* ให้รวมไฟล์อัตโนมัติ
part_files = sorted([f for f in os.listdir('/content') if f.startswith('dataset.part_')])
if part_files and not os.path.exists(dataset_zip):
    print(f"กำลังรวมไฟล์ {len(part_files)} พาร์ท เข้าเป็น {dataset_zip}...")
    with open(dataset_zip, 'wb') as outfile:
        for pf in part_files:
            p_path = os.path.join('/content', pf)
            with open(p_path, 'rb') as infile:
                outfile.write(infile.read())
    print(f"[OK] รวมไฟล์สำเร็จ: {dataset_zip} ({os.path.getsize(dataset_zip)/(1024*1024):.2f} MB)")

# แตกไฟล์ Dataset
if os.path.exists(extract_dir):
    shutil.rmtree(extract_dir)
os.makedirs(extract_dir, exist_ok=True)

if os.path.exists(dataset_zip):
    print(f"กำลังแตกไฟล์ {dataset_zip}...")
    with zipfile.ZipFile(dataset_zip, 'r') as zip_ref:
        for member in zip_ref.namelist():
            normalized = member.replace('\\', '/')
            target_path = os.path.join(extract_dir, normalized)
            if normalized.endswith('/'):
                os.makedirs(target_path, exist_ok=True)
            else:
                os.makedirs(os.path.dirname(target_path), exist_ok=True)
                with zip_ref.open(member) as src, open(target_path, 'wb') as dst:
                    dst.write(src.read())
    print("✅ แตกไฟล์และจัดโครงสร้าง Dataset สำเร็จเรียบร้อย!")
else:
    print("⚠️ ไม่พบไฟล์ dataset.zip ใน /content/ กรุณาอัปโหลดไฟล์ dataset.zip หรือเชื่อมต่อ Google Drive")

## 🔍 ขั้นตอนที่ 4: ตรวจสอบความถูกต้องของ Dataset และ `data.yaml`

In [ ]:
yaml_path = "/content/dataset/data.yaml"

# ตรวจสอบและสร้าง/แก้ไข data.yaml ให้ถูกต้อง
yaml_content = """path: /content/dataset
train: train/images
val: valid/images
test: test/images

nc: 2
names: ['with-helmet', 'without-helmet']
"""

with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(yaml_content.strip())

train_img_dir = "/content/dataset/train/images"
valid_img_dir = "/content/dataset/valid/images"
test_img_dir = "/content/dataset/test/images"

print("--- ข้อมูล Dataset ---")
print(f"📁 รูปภาพสำหรับ Train: {len(os.listdir(train_img_dir)) if os.path.exists(train_img_dir) else 0} รูป")
print(f"📁 รูปภาพสำหรับ Validation: {len(os.listdir(valid_img_dir)) if os.path.exists(valid_img_dir) else 0} รูป")
print(f"📁 รูปภาพสำหรับ Test: {len(os.listdir(test_img_dir)) if os.path.exists(test_img_dir) else 0} รูป")
print("----------------------")
with open(yaml_path, "r", encoding="utf-8") as f:
    print(f.read())

## 🎯 ขั้นตอนที่ 5: เริ่มเทรนโมเดล YOLOv8n (200 Epochs)

In [ ]:
from ultralytics import YOLO

# โหลด Base Model YOLOv8n
model = YOLO('yolov8n.pt')

# สั่งเริ่มการเทรน
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=200,          # จำนวนรอบการเทรน
    imgsz=640,           # ขนาดภาพ
    batch=16,            # Batch size
    patience=50,         # Early stopping ถ้านิ่งเกิน 50 epochs
    device=0 if torch.cuda.is_available() else 'cpu',
    project='/content/runs/detect',
    name='train',
    exist_ok=True,
    verbose=True,
    save=True
)

print("\n🎉 เทรนโมเดลเสร็จสมบูรณ์เรียบร้อยแล้ว!")

## 📊 ขั้นตอนที่ 6: แสดงผลกราฟความแม่นยำ (Results & Confusion Matrix)

In [ ]:
from IPython.display import Image, display

results_png = "/content/runs/detect/train/results.png"
cm_png = "/content/runs/detect/train/confusion_matrix.png"

if os.path.exists(results_png):
    print("📈 กราฟผลการเทรน (Loss & Metrics):")
    display(Image(results_png))

if os.path.exists(cm_png):
    print("🎯 Confusion Matrix (ความแม่นยำแต่ละคลาส):")
    display(Image(cm_png))

## ⚙️ ขั้นตอนที่ 7: แปลงโมเดลเป็น NCNN Format (สำหรับ PC & Mobile App)

In [ ]:
weights_path = "/content/runs/detect/train/weights/best.pt"

# 1. Copy best.pt ไว้ที่ /content/
shutil.copy(weights_path, "/content/best.pt")
print(f"✅ บันทึก /content/best.pt สำเร็จ ({os.path.getsize('/content/best.pt')/(1024*1024):.2f} MB)")

# 2. Export เป็น NCNN
trained_model = YOLO(weights_path)
ncnn_dir = trained_model.export(format="ncnn", imgsz=640)
print(f"✅ Export NCNN ไปที่: {ncnn_dir}")

# 3. บีบอัดเป็น ZIP สำหรับดาวน์โหลด
zip_target = "/content/best_ncnn_model.zip"
with zipfile.ZipFile(zip_target, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(ncnn_dir):
        for f in files:
            full_path = os.path.join(root, f)
            arcname = os.path.relpath(full_path, ncnn_dir)
            zipf.write(full_path, arcname)

print(f"✅ บีบอัดไฟล์ NCNN สำเร็จ: {zip_target} ({os.path.getsize(zip_target)/(1024*1024):.2f} MB)")

## 💾 ขั้นตอนที่ 8: ดาวน์โหลดโมเดลและไฟล์ผลลัพธ์ลงเครื่องคอมพิวเตอร์

In [ ]:
from google.colab import files

print("กำลังดาวน์โหลด best.pt และ best_ncnn_model.zip ลงเครื่อง...")
files.download("/content/best.pt")
files.download("/content/best_ncnn_model.zip")
if os.path.exists("/content/runs/detect/train/results.png"):
    files.download("/content/runs/detect/train/results.png")
if os.path.exists("/content/runs/detect/train/confusion_matrix.png"):
    files.download("/content/runs/detect/train/confusion_matrix.png")

print("🎉 ดาวน์โหลดไฟล์ทั้งหมดเรียบร้อยแล้ว!")